# PREPROCESSING TEKS ULASAN E-COMMERCE

**Nama:** Fitri Simah Bengi

**NIM:** 25146043

**Mata Kuliah:** Natural Language Processing

In [25]:
from IPython.display import display, HTML

display(HTML("""
<style>
.jp-OutputArea-output,
.jp-RenderedText,
.jp-RenderedHTMLCommon,
.jp-RenderedHTML,
.output,
pre {
    text-align: left !important;
}

.jp-RenderedHTMLCommon table,
.jp-RenderedHTMLCommon th,
.jp-RenderedHTMLCommon td,
table th,
table td {
    text-align: left !important;
    vertical-align: top !important;
}
</style>
"""))

In [13]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
import numpy as np
import re
import requests

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("Library berhasil dipanggil!")

Library berhasil dipanggil!


In [14]:
url = "https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment/resolve/main/labeledReview.datasetFix.json"

response = requests.get(url)

print("Status:", response.status_code)

Status: 200


In [15]:
data = response.json()

df = pd.DataFrame(data)

print("Jumlah baris dan kolom:", df.shape)
print("\nNama kolom:")
print(df.columns.tolist())

display(df.head())

Jumlah baris dan kolom: (11606, 3)

Nama kolom:
['review', 'sentimen', 'translate']


,review,sentimen,translate
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,0,"sound and mic satisfied. I only ordered the one that was on and I made sure that it clicked again, but the one that came didn't turn on"
1,Good. Mudah-mudahan awet. Makasih,1,good. hopefully durable. Thanks
2,"kondisi dus robek2, tolong di perhatikan lagi",0,"the condition of the box is torn, please pay attention again"
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,0,"product quality is good. affordable prices and very fast delivery. I've already ordered here, it's okay, but unfortunately, if you make a gift, you can't bring it with a greeting card or wrap it in gift paper :("
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",0,"overall is good, the buttons all work, but the packaging is lacking because it's dented and torn a little"


In [16]:
print("Jumlah seluruh data:", len(df))
print("Jumlah kolom:", len(df.columns))

print("\nDistribusi Sentimen:")
print(df["sentimen"].value_counts())

print("\nPersentase Sentimen:")
print(df["sentimen"].value_counts(normalize=True).mul(100).round(2))

Jumlah seluruh data: 11606
Jumlah kolom: 3

Distribusi Sentimen:
sentimen
0    5803
1    5803
Name: count, dtype: int64

Persentase Sentimen:
sentimen
0    50.0
1    50.0
Name: proportion, dtype: float64


In [17]:
df["case_folding"] = df["review"].astype(str).str.lower()

display(df[["review", "case_folding"]].head(10))

,review,case_folding
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara dan mic puas. hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala
1,Good. Mudah-mudahan awet. Makasih,good. mudah-mudahan awet. makasih
2,"kondisi dus robek2, tolong di perhatikan lagi","kondisi dus robek2, tolong di perhatikan lagi"
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk baik. harga terjangkau dan pengirimannya cepet banget. udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit","overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit"
5,"Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.","produk ini original, dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller."
6,kualitas sangat jelek. bocor,kualitas sangat jelek. bocor
7,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka","terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka"
8,"Body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli CM 1659 penasaran Aja, yang cm 1579 udah mau rusak mesin nya, Harga lebih murah 4k dari cm 1579 Fast delivery, courier Ramah sangat, ty seller","body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli cm 1659 penasaran aja, yang cm 1579 udah mau rusak mesin nya, harga lebih murah 4k dari cm 1579 fast delivery, courier ramah sangat, ty seller"
9,pesanan di terima tanggal 27 January 2021 tetapi baru bisa saya buka pagi ini tanggal 29 January 2021 pukul 06.00 dan ternyata tab SKETSA ADVAN tidak bisa di nyalakan sama sekali dan di cas juga posisi Tab tidak bisa menyala,pesanan di terima tanggal 27 january 2021 tetapi baru bisa saya buka pagi ini tanggal 29 january 2021 pukul 06.00 dan ternyata tab sketsa advan tidak bisa di nyalakan sama sekali dan di cas juga posisi tab tidak bisa menyala


In [18]:
def cleaning_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["cleaning"] = df["case_folding"].apply(cleaning_text)

display(df[["case_folding", "cleaning"]].head(10))

,case_folding,cleaning
0,suara dan mic puas. hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara dan mic puas hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala
1,good. mudah-mudahan awet. makasih,good mudahmudahan awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek tolong di perhatikan lagi
3,kualitas produk baik. harga terjangkau dan pengirimannya cepet banget. udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk baik harga terjangkau dan pengirimannya cepet banget udh x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado
4,"overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol berfungsi semuanya tapi packaging kurang karena sudah penyot dan robek sedikit
5,"produk ini original, dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.",produk ini original dan kualitasnya sangat baik kecepatan pengiriman sanagt cepat recommended seller
6,kualitas sangat jelek. bocor,kualitas sangat jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka",terimakasih banyak ka ya barang sudah diterima dengan baik sukses terus ka
8,"body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli cm 1659 penasaran aja, yang cm 1579 udah mau rusak mesin nya, harga lebih murah 4k dari cm 1579 fast delivery, courier ramah sangat, ty seller",body nya lebih berat dari cm spiral nya lebih tipis dibanding cm semoga awet beli cm penasaran aja yang cm udah mau rusak mesin nya harga lebih murah k dari cm fast delivery courier ramah sangat ty seller
9,pesanan di terima tanggal 27 january 2021 tetapi baru bisa saya buka pagi ini tanggal 29 january 2021 pukul 06.00 dan ternyata tab sketsa advan tidak bisa di nyalakan sama sekali dan di cas juga posisi tab tidak bisa menyala,pesanan di terima tanggal january tetapi baru bisa saya buka pagi ini tanggal january pukul dan ternyata tab sketsa advan tidak bisa di nyalakan sama sekali dan di cas juga posisi tab tidak bisa menyala


In [19]:
df["tokenisasi"] = df["cleaning"].apply(lambda teks: teks.split())

display(df[["cleaning", "tokenisasi"]].head(10))

,cleaning,tokenisasi
0,suara dan mic puas hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,"[suara, dan, mic, puas, hanya, saya, pesan, yg, nyala, dan, sudah, sy, pastikan, lg, klik, yg, nyala, tp, yg, datang, ga, nyala]"
1,good mudahmudahan awet makasih,"[good, mudahmudahan, awet, makasih]"
2,kondisi dus robek tolong di perhatikan lagi,"[kondisi, dus, robek, tolong, di, perhatikan, lagi]"
3,kualitas produk baik harga terjangkau dan pengirimannya cepet banget udh x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado,"[kualitas, produk, baik, harga, terjangkau, dan, pengirimannya, cepet, banget, udh, x, order, disini, oke, lah, tp, sayangnya, klo, buat, kado, gabisa, nitip, pake, kartu, ucapan, atau, dibungkus, pake, kertas, kado]"
4,overall bagus tombol berfungsi semuanya tapi packaging kurang karena sudah penyot dan robek sedikit,"[overall, bagus, tombol, berfungsi, semuanya, tapi, packaging, kurang, karena, sudah, penyot, dan, robek, sedikit]"
5,produk ini original dan kualitasnya sangat baik kecepatan pengiriman sanagt cepat recommended seller,"[produk, ini, original, dan, kualitasnya, sangat, baik, kecepatan, pengiriman, sanagt, cepat, recommended, seller]"
6,kualitas sangat jelek bocor,"[kualitas, sangat, jelek, bocor]"
7,terimakasih banyak ka ya barang sudah diterima dengan baik sukses terus ka,"[terimakasih, banyak, ka, ya, barang, sudah, diterima, dengan, baik, sukses, terus, ka]"
8,body nya lebih berat dari cm spiral nya lebih tipis dibanding cm semoga awet beli cm penasaran aja yang cm udah mau rusak mesin nya harga lebih murah k dari cm fast delivery courier ramah sangat ty seller,"[body, nya, lebih, berat, dari, cm, spiral, nya, lebih, tipis, dibanding, cm, semoga, awet, beli, cm, penasaran, aja, yang, cm, udah, mau, rusak, mesin, nya, harga, lebih, murah, k, dari, cm, fast, delivery, courier, ramah, sangat, ty, seller]"
9,pesanan di terima tanggal january tetapi baru bisa saya buka pagi ini tanggal january pukul dan ternyata tab sketsa advan tidak bisa di nyalakan sama sekali dan di cas juga posisi tab tidak bisa menyala,"[pesanan, di, terima, tanggal, january, tetapi, baru, bisa, saya, buka, pagi, ini, tanggal, january, pukul, dan, ternyata, tab, sketsa, advan, tidak, bisa, di, nyalakan, sama, sekali, dan, di, cas, juga, posisi, tab, tidak, bisa, menyala]"


In [20]:
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

df["stopword_removal"] = df["tokenisasi"].apply(
    lambda tokens: [kata for kata in tokens if kata not in stopwords]
)

display(df[["tokenisasi", "stopword_removal"]].head(10))

,tokenisasi,stopword_removal
0,"[suara, dan, mic, puas, hanya, saya, pesan, yg, nyala, dan, sudah, sy, pastikan, lg, klik, yg, nyala, tp, yg, datang, ga, nyala]","[suara, mic, puas, pesan, yg, nyala, sy, pastikan, lg, klik, yg, nyala, tp, yg, datang, ga, nyala]"
1,"[good, mudahmudahan, awet, makasih]","[good, mudahmudahan, awet, makasih]"
2,"[kondisi, dus, robek, tolong, di, perhatikan, lagi]","[kondisi, dus, robek, perhatikan]"
3,"[kualitas, produk, baik, harga, terjangkau, dan, pengirimannya, cepet, banget, udh, x, order, disini, oke, lah, tp, sayangnya, klo, buat, kado, gabisa, nitip, pake, kartu, ucapan, atau, dibungkus, pake, kertas, kado]","[kualitas, produk, baik, harga, terjangkau, pengirimannya, cepet, banget, udh, x, order, disini, oke, lah, tp, sayangnya, klo, buat, kado, gabisa, nitip, pake, kartu, ucapan, dibungkus, pake, kertas, kado]"
4,"[overall, bagus, tombol, berfungsi, semuanya, tapi, packaging, kurang, karena, sudah, penyot, dan, robek, sedikit]","[overall, bagus, tombol, berfungsi, semuanya, packaging, kurang, penyot, robek, sedikit]"
5,"[produk, ini, original, dan, kualitasnya, sangat, baik, kecepatan, pengiriman, sanagt, cepat, recommended, seller]","[produk, original, kualitasnya, sangat, baik, kecepatan, pengiriman, sanagt, cepat, recommended, seller]"
6,"[kualitas, sangat, jelek, bocor]","[kualitas, sangat, jelek, bocor]"
7,"[terimakasih, banyak, ka, ya, barang, sudah, diterima, dengan, baik, sukses, terus, ka]","[terimakasih, banyak, ka, barang, diterima, baik, sukses, terus, ka]"
8,"[body, nya, lebih, berat, dari, cm, spiral, nya, lebih, tipis, dibanding, cm, semoga, awet, beli, cm, penasaran, aja, yang, cm, udah, mau, rusak, mesin, nya, harga, lebih, murah, k, dari, cm, fast, delivery, courier, ramah, sangat, ty, seller]","[body, nya, lebih, berat, cm, spiral, nya, lebih, tipis, dibanding, cm, semoga, awet, beli, cm, penasaran, aja, cm, udah, mau, rusak, mesin, nya, harga, lebih, murah, k, cm, fast, delivery, courier, ramah, sangat, ty, seller]"
9,"[pesanan, di, terima, tanggal, january, tetapi, baru, bisa, saya, buka, pagi, ini, tanggal, january, pukul, dan, ternyata, tab, sketsa, advan, tidak, bisa, di, nyalakan, sama, sekali, dan, di, cas, juga, posisi, tab, tidak, bisa, menyala]","[pesanan, terima, tanggal, january, baru, buka, pagi, tanggal, january, pukul, ternyata, tab, sketsa, advan, nyalakan, sama, sekali, cas, posisi, tab, menyala]"


In [21]:
factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

# Mempercepat proses dengan menyimpan hasil kata yang sudah diproses
stem_cache = {}

def stemming_tokens(tokens):
    hasil = []
    for kata in tokens:
        if kata not in stem_cache:
            stem_cache[kata] = stemmer.stem(kata)
        hasil.append(stem_cache[kata])
    return hasil

df["stemming"] = df["stopword_removal"].apply(stemming_tokens)

display(df[["stopword_removal", "stemming"]].head(10))

,stopword_removal,stemming
0,"[suara, mic, puas, pesan, yg, nyala, sy, pastikan, lg, klik, yg, nyala, tp, yg, datang, ga, nyala]","[suara, mic, puas, pesan, yg, nyala, sy, pasti, lg, klik, yg, nyala, tp, yg, datang, ga, nyala]"
1,"[good, mudahmudahan, awet, makasih]","[good, mudahmudahan, awet, makasih]"
2,"[kondisi, dus, robek, perhatikan]","[kondisi, dus, robek, perhati]"
3,"[kualitas, produk, baik, harga, terjangkau, pengirimannya, cepet, banget, udh, x, order, disini, oke, lah, tp, sayangnya, klo, buat, kado, gabisa, nitip, pake, kartu, ucapan, dibungkus, pake, kertas, kado]","[kualitas, produk, baik, harga, jangkau, kirim, cepet, banget, udh, x, order, sini, oke, lah, tp, sayang, klo, buat, kado, gabisa, nitip, pake, kartu, ucap, bungkus, pake, kertas, kado]"
4,"[overall, bagus, tombol, berfungsi, semuanya, packaging, kurang, penyot, robek, sedikit]","[overall, bagus, tombol, fungsi, semua, packaging, kurang, penyot, robek, sedikit]"
5,"[produk, original, kualitasnya, sangat, baik, kecepatan, pengiriman, sanagt, cepat, recommended, seller]","[produk, original, kualitas, sangat, baik, cepat, kirim, sanagt, cepat, recommended, seller]"
6,"[kualitas, sangat, jelek, bocor]","[kualitas, sangat, jelek, bocor]"
7,"[terimakasih, banyak, ka, barang, diterima, baik, sukses, terus, ka]","[terimakasih, banyak, ka, barang, terima, baik, sukses, terus, ka]"
8,"[body, nya, lebih, berat, cm, spiral, nya, lebih, tipis, dibanding, cm, semoga, awet, beli, cm, penasaran, aja, cm, udah, mau, rusak, mesin, nya, harga, lebih, murah, k, cm, fast, delivery, courier, ramah, sangat, ty, seller]","[body, nya, lebih, berat, cm, spiral, nya, lebih, tipis, banding, cm, moga, awet, beli, cm, penasaran, aja, cm, udah, mau, rusak, mesin, nya, harga, lebih, murah, k, cm, fast, delivery, courier, ramah, sangat, ty, seller]"
9,"[pesanan, terima, tanggal, january, baru, buka, pagi, tanggal, january, pukul, ternyata, tab, sketsa, advan, nyalakan, sama, sekali, cas, posisi, tab, menyala]","[pesan, terima, tanggal, january, baru, buka, pagi, tanggal, january, pukul, nyata, tab, sketsa, advan, nyala, sama, sekali, cas, posisi, tab, nyala]"


In [23]:
hasil_10 = pd.DataFrame({
    "Teks Sebelum": df["review"].head(10).astype(str),
    "Teks Sesudah": df["stemming"].head(10).apply(
        lambda tokens: " ".join(tokens)
    )
})

tabel_rapi = (
    hasil_10.style
    .set_properties(**{
        "text-align": "left",
        "white-space": "normal",
        "word-wrap": "break-word",
        "min-width": "350px",
        "max-width": "500px"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [("text-align", "left")]
        },
        {
            "selector": "table",
            "props": [
                ("width", "100%"),
                ("table-layout", "fixed")
            ]
        }
    ])
)

display(tabel_rapi)

,Teks Sebelum,Teks Sesudah
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg datang ga nyala
1,Good. Mudah-mudahan awet. Makasih,good mudahmudahan awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek perhati
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk baik harga jangkau kirim cepet banget udh x order sini oke lah tp sayang klo buat kado gabisa nitip pake kartu ucap bungkus pake kertas kado
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol fungsi semua packaging kurang penyot robek sedikit
5,"Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.",produk original kualitas sangat baik cepat kirim sanagt cepat recommended seller
6,kualitas sangat jelek. bocor,kualitas sangat jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka",terimakasih banyak ka barang terima baik sukses terus ka
8,"Body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli CM 1659 penasaran Aja, yang cm 1579 udah mau rusak mesin nya, Harga lebih murah 4k dari cm 1579 Fast delivery, courier Ramah sangat, ty seller",body nya lebih berat cm spiral nya lebih tipis banding cm moga awet beli cm penasaran aja cm udah mau rusak mesin nya harga lebih murah k cm fast delivery courier ramah sangat ty seller
9,pesanan di terima tanggal 27 January 2021 tetapi baru bisa saya buka pagi ini tanggal 29 January 2021 pukul 06.00 dan ternyata tab SKETSA ADVAN tidak bisa di nyalakan sama sekali dan di cas juga posisi Tab tidak bisa menyala,pesan terima tanggal january baru buka pagi tanggal january pukul nyata tab sketsa advan nyala sama sekali cas posisi tab nyala


## Pengaruh Preprocessing terhadap Kualitas Teks

Preprocessing merupakan tahapan penting dalam pengolahan data teks sebelum dilakukan analisis lebih lanjut. Pada penelitian ini, preprocessing dilakukan melalui beberapa tahapan, yaitu case folding, cleaning, tokenisasi, stopword removal, dan stemming.

Setiap tahapan memberikan pengaruh terhadap kualitas teks, antara lain:

1. **Case Folding:** Mengubah seluruh huruf menjadi huruf kecil sehingga penulisan kata menjadi seragam.
2. **Cleaning:** Menghapus URL, angka, tanda baca, dan karakter khusus yang tidak diperlukan.
3. **Tokenisasi:** Memecah kalimat menjadi kata-kata agar lebih mudah dianalisis.
4. **Stopword Removal:** Menghapus kata-kata umum yang dianggap kurang memberikan informasi penting.
5. **Stemming:** Mengubah kata berimbuhan menjadi kata dasar sehingga bentuk kata menjadi lebih sederhana.

Dengan adanya preprocessing, data ulasan menjadi lebih bersih, terstruktur, dan mudah diolah untuk proses analisis sentimen. Namun, penghapusan angka dan kata tertentu juga dapat menyebabkan sebagian informasi dalam ulasan hilang.